# Interactive feature matching

In [ ]:
import sys
sys.path.append('..')

import numpy as np
from multiview_stitcher import spatial_image_utils as si_utils

from muvis_align.image.ome_tiff_helper import save_tiff
from muvis_align.MVSRegistration import MVSRegistration
from muvis_align.image.util import get_sim_position_final, get_sim_physical_size, draw_keypoints_matches, \
    get_overlap_images, sims_from_sims_or_msims

## Setup parameters

In [ ]:
input_path = ['../data/S000/S000_000_000.ome.zarr', '../data/S001/S001_000_001.ome.zarr']
extra_metadata = '../aligned_stitched_mappings1.json'
output_path = '../../output/'

## Initialise muvis-align, initialise sims, and pre-process

In [ ]:
reg = MVSRegistration(input_path=input_path, extra_metadata=extra_metadata,
                      output_path=output_path, ui='mpl', debug=True)
reg.file_labels = ['S000_000_000', 'S001_000_001']
reg.init_data()
msims = reg.msims
norm_msims, register_indices, _ = reg.preprocess(msims)
sims = sims_from_sims_or_msims(msims)
norm_sims = sims_from_sims_or_msims(norm_msims)

for label, position, sim in zip(reg.file_labels, reg.positions, sims):
    print(label, get_sim_position_final(sim, position), get_sim_physical_size(sim))

## Get overlap between tiles

In [ ]:
%matplotlib inline
overlap1, overlap2, _ = get_overlap_images(norm_sims[0], norm_sims[1], reg.source_transform_key)
overlap1, overlap2 = overlap1.squeeze().compute(), overlap2.squeeze().compute()
print('Overlap size [um]:', overlap1.shape * si_utils.get_spacing_from_sim(sims[0], asarray=True))

draw_keypoints_matches(overlap1, [], overlap2, [])

## Interactive: Set up feature extraction and matching, and show results

### You can modify the parameters and rerun this cell to update results, e.g.:

**transform_type**: affine, euclidean, translation

**name**: sift, orb

In [ ]:
%matplotlib inline

register_params = {
	'pairing': 'orthogonal',
	'transform_type': 'rigid',
	'method': 'orb',
	'gaussian_sigma': 2.5,
	'normalisation': True,
	'max_keypoints': 5000,
	'inlier_threshold_factor': 0.05,
	'max_trials': 1000,
	'ransac_iterations': 3,
	'n_parallel_pairwise_regs': 1,
}

reg_method, pairwise_reg_func, pairwise_reg_func_kwargs = (
    reg.create_registration_method(sims[0], params=register_params))
result = pairwise_reg_func(overlap1, overlap2)
print('transform:\n', result['affine_matrix'])

## Run registration on all tiles, showing results corresponding to each tile overlap

In [ ]:
%matplotlib inline
reg.register(register_msims=norm_msims, register_indices=register_indices, params=register_params)